In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DB_PATH = "restaurant.db"          # <-- поставь свой путь
OUT_CSV = "joined_data.csv"


def extract_joined_data(db_path: str) -> pd.DataFrame:
    con = sqlite3.connect(db_path)

    # ВНИМАНИЕ: проверь названия колонок в твоей БД.
    # Ниже самый типичный вариант:
    # products: id, name, price
    # orders: id, created_at (или datetime)
    # order_items: order_id, product_id, quantity
    query = """
    SELECT
        oi.*,
        p.name AS product_name,
        p.price AS product_price,
        o.created_at AS order_datetime
    FROM order_items oi
    JOIN products p
        ON p.id = oi.product_id
    JOIN orders o
        ON o.id = oi.order_id
    ;
    """

    df = pd.read_sql_query(query, con)
    con.close()
    return df


df = extract_joined_data(DB_PATH)

# Quick check
print("Rows:", len(df))
# должно быть 74818

df.to_csv(OUT_CSV, index=False)
df = pd.read_csv(OUT_CSV)

df.head()


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top10_qty = (
    df.groupby("product_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure()
plt.pie(top10_qty.values, labels=top10_qty.index, autopct="%1.1f%%")
plt.title("Top 10 products by quantity sold")
plt.tight_layout()
plt.show()


# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["product_price"] * df["quantity"]

top10_revenue = (
    df.groupby("product_name")["item_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure()
plt.pie(top10_revenue.values, labels=top10_revenue.index, autopct="%1.1f%%")
plt.title("Top 10 products by revenue (item_price)")
plt.tight_layout()
plt.show()


# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
# Приводим к datetime (если строка)
df["order_datetime"] = pd.to_datetime(df["order_datetime"], errors="coerce")

df["order_hour"] = df["order_datetime"].dt.hour

income_by_hour = (
    df.groupby("order_hour")["item_price"]
    .sum()
    .reindex(range(24), fill_value=0)
)

plt.figure()
plt.bar(income_by_hour.index, income_by_hour.values)
plt.title("Total income by order hour")
plt.xlabel("Hour of day")
plt.ylabel("Total income")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df["day_of_week"] = df["order_datetime"].dt.day_name()

days_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
              "Friday", "Saturday", "Sunday"]

income_by_day = (
    df.groupby("day_of_week")["item_price"]
    .sum()
    .reindex(days_order, fill_value=0)
)

plt.figure()
plt.bar(income_by_day.index, income_by_day.values)
plt.title("Total income by day of week")
plt.xlabel("Day of week")
plt.ylabel("Total income")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
